# Memory Transfer

We'll now examine how to generate and capture signals with the RF hardware integrated into the RFSoC. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
import numpy.random as random

from acadia.system import Acadia, StreamConfiguration
from acadia.channel import Channel
from acadia.arrays import ProceduralWaveform

In [2]:
acadia = Acadia()

arr1 = acadia.PLDDR0Array(size=16*1000)
arr2 = acadia.PLDDR1Array(size=16*1000)

transfer_configuration = StreamConfiguration("bulk", module="bulk", acadia=acadia)  

In [3]:
# Create a sequence for the sequencer
def sequence(a):
    a.memcpy(arr1, arr2, transfer_configuration)

acadia.attach()
acadia.configure_stream(transfer_configuration)

arr1.memory[:] = np.frombuffer(random.default_rng(1278).bytes(arr1.byte_length()), dtype=np.uint8)
arr2.memory[:] = np.zeros(arr2.byte_length(), dtype=np.uint8)

print(f"{arr1.memory[:16]} ... {arr1.memory[-16:]}")
print(f"{arr2.memory[:16]} ... {arr2.memory[-16:]}")

acadia.run(sequence)
time.sleep(0.1)
acadia.sequencer_halt()

print(f"{arr1.memory[:16]} ... {arr1.memory[-16:]}")
print(f"{arr2.memory[:16]} ... {arr2.memory[-16:]}")

[ 24 223 123 175  49 114 178 145 143  33  24 131 102 160 231   3] ... [ 15 148  92 148  49  56 130  27 252 116 146 206 207  56  65 170]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] ... [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[ 24 223 123 175  49 114 178 145 143  33  24 131 102 160 231   3] ... [ 15 148  92 148  49  56 130  27 252 116 146 206 207  56  65 170]
[ 24 223 123 175  49 114 178 145 143  33  24 131 102 160 231   3] ... [ 15 148  92 148  49  56 130  27 252 116 146 206 207  56  65 170]


In [5]:
np.all(arr1.memory == arr2.memory)

True